## Import libraries and load Gemini API key


In [ ]:
!pip install openai

In [ ]:
# Import necessary libraries
from openai import OpenAI
from google.genai import types
from google.colab import userdata
import pandas as pd
import re
import tiktoken
from tqdm import tqdm
tqdm.pandas()

# Load OpenAI API key from Colab secure variables
OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
# Initialize OpenAI client
client = OpenAI(api_key=OPENAI_API_KEY)
# Select model
MODEL = "gpt-5.2"
# MODEL = "gpt-5-mini"

## Load dataset


In [2]:
# connect Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [11]:
# Load individual datasets per task
path = "/content/drive/MyDrive/tfg/corpusMentalRiskES/splits/LLM/"
path_tara5 = path + "tara5/"

df_anx = pd.read_csv(path + "anxiety_llm.csv", sep=",")
df_dep = pd.read_csv(path + "depression_llm.csv", sep=",")
df_ed = pd.read_csv(path + "ed_llm.csv", sep=",")
df_multi = pd.read_csv(path + "multiclass_llm.csv", sep=",")

df_anx_tara5 = pd.read_csv(path_tara5 + "anxiety_llm_tara5.csv")
df_dep_tara5 = pd.read_csv(path_tara5 + "depression_llm_tara5.csv")
df_ed_tara5  = pd.read_csv(path_tara5 + "ed_llm_tara5.csv")
df_multi_tara5 = pd.read_csv(path_tara5 + "multiclass_llm_tara5.csv")

# Select classification task: anx | dep | ed | multi
task = "multi" # <--- Change here

# Assign the dataframe corresponding to the selected task
if task == "anx":
  df = df_anx
  df_tara5 = df_anx_tara5
elif task == "dep":
  df = df_dep
  df_tara5 = df_dep_tara5
elif task == "ed":
  df = df_ed
  df_tara5 = df_ed_tara5
elif task == "multi":
  df = df_multi
  df_tara5 = df_multi_tara5

## Function to build the prompt


The text to be classified is added to a fixed prompt that changes according to the task.

> Note: Since the study focuses on Spanish texts, the prompts passed to the models remain in Spanish.


In [4]:
def build_prompt(text, task):
  """ Dynamic construction of the prompt according to the task.
  Returns the complete text to be sent to the model.
  """

  if task == "anx":
    return f"""
Vas a recibir un texto que corresponde a una serie de mensajes escritos por una persona en una red social.
El texto puede o no reflejar indicios de ansiedad.

La ansiedad se asocia a una preocupación persistente y difícil de controlar, acompañada de nerviosismo, inquietud interna y una sensación continua de amenaza o anticipación negativa. Estas experiencias suelen generar malestar emocional sostenido y dificultad para relajarse.

Tu tarea es clasificar el texto únicamente en una de las siguientes categorías:

1: el texto muestra indicios de ansiedad
0: el texto no muestra indicios de ansiedad

El contenido de los mensajes se proporciona a continuación entre comillas triples:

\"\"\"{text}\"\"\"

Responde solo con el número correspondiente.
"""
  elif task == "dep":
    return f"""
Vas a recibir un texto que corresponde a una serie de mensajes escritos por una persona en una red social.
El texto puede o no reflejar indicios de depresión.

La depresión se asocia a un estado persistente de tristeza, apatía o vacío emocional, acompañado de una disminución del interés o la capacidad de disfrute en actividades habituales. Estas experiencias pueden incluir pensamientos negativos recurrentes, fatiga emocional y una percepción pesimista de uno mismo o del futuro.

Tu tarea es clasificar el texto únicamente en una de las siguientes categorías:

1: el texto muestra indicios de depresión
0: el texto no muestra indicios de depresión

El contenido de los mensajes se proporciona a continuación entre comillas triples:

\"\"\"{text}\"\"\"

Responde solo con el número correspondiente.
"""
  elif task == "ed":
    return f"""
Vas a recibir un texto que corresponde a una serie de mensajes escritos por una persona en una red social.
El texto puede o no reflejar indicios de un trastorno de la conducta alimentaria.

Los trastornos de la conducta alimentaria se asocian a una preocupación persistente por el peso corporal, la alimentación o la imagen corporal, que puede manifestarse mediante pensamientos rígidos o negativos sobre la comida, el cuerpo o el control del peso. Estas preocupaciones suelen generar malestar emocional, culpa, ansiedad en torno a la alimentación y conductas de evitación o control.

Tu tarea es clasificar el texto únicamente en una de las siguientes categorías:

1: el texto muestra indicios de un trastorno de la conducta alimentaria
0: el texto no muestra indicios de un trastorno de la conducta alimentaria

El contenido de los mensajes se proporciona a continuación entre comillas triples:

\"\"\"{text}\"\"\"

Responde solo con el número correspondiente.
"""
  else:
    return f"""
Vas a recibir un texto que corresponde a una serie de mensajes escritos por una persona en una red social.
El texto puede reflejar indicios de uno de los siguientes estados psicológicos, o no reflejar ninguno de ellos.

Ansiedad: se asocia a una preocupación persistente y difícil de controlar, acompañada de nerviosismo, inquietud interna y una sensación continua de amenaza o anticipación negativa, generando malestar emocional y dificultad para relajarse.

Depresión: se asocia a un estado persistente de tristeza, apatía o vacío emocional, junto con una disminución del interés o la capacidad de disfrute en actividades habituales, pensamientos negativos recurrentes y una visión pesimista de uno mismo o del futuro.

Trastornos de la conducta alimentaria: se asocian a una preocupación persistente por el peso, la alimentación o la imagen corporal, que puede manifestarse mediante pensamientos rígidos o negativos sobre la comida o el cuerpo, así como malestar emocional, culpa o ansiedad en torno a la alimentación.

Tu tarea es clasificar el texto únicamente en una de las siguientes categorías:

0: el texto no muestra indicios de ninguno de los estados descritos
1: el texto muestra indicios de ansiedad
2: el texto muestra indicios de depresión
3: el texto muestra indicios de un trastorno de la conducta alimentaria

El contenido de los mensajes se proporciona a continuación entre comillas triples:

\"\"\"{text}\"\"\"

Responde solo con el número correspondiente.
"""

## Calculate the number of tokens needed to process the dataset


In [ ]:
# Obtain the tokenizer associated with the model
encoding = tiktoken.get_encoding("cl100k_base")

def total_tokens_dataset(df, task):
  """ Calculates the total number of input tokens
  for a complete dataset, using the model's tokenizer
  """
  total_tokens = 0

  for text in df["texto"]:
    prompt = build_prompt(text, task)
    tokens = len(encoding.encode(prompt))
    total_tokens += tokens

  return total_tokens


# Token calculation per dataset
total_anx = total_tokens_dataset(df_anx, "anx")
print("ANXIETY DATASET")
print("Tokens:", total_anx)
print("Average per text:", total_anx / len(df_anx))

total_dep = total_tokens_dataset(df_dep, "dep")
print()
print("DEPRESSION DATASET")
print("Tokens:", total_dep)
print("Average per text:", total_dep / len(df_dep))

total_ed = total_tokens_dataset(df_ed, "ed")
print()
print("ED DATASET")
print("Tokens:", total_ed)
print("Average per text:", total_ed / len(df_ed))

total_multi = total_tokens_dataset(df_multi, "multi")
print()
print("MULTICLASS DATASET")
print("Tokens:", total_multi)
print("Average per text:", total_multi / len(df_multi))

# Total sum of tokens needed for the entire experiment
print()
print(f"Total tokens: {total_anx + total_dep + total_ed + total_multi}")

DATASET ANSIEDAD
Tokens: 511422
Media por texto: 1022.844

DATASET DEPRESIÓN
Tokens: 433676
Media por texto: 869.0901803607214

DATASET ED
Tokens: 299544
Media por texto: 894.1611940298508

DATASET MULTICLASE
Tokens: 1464529
Media por texto: 1097.8478260869565

Total de tokens: 2709171


## Zero-shot classification with GPT models

In [ ]:
def classify_text(text, task):
    """ Classification function using OpenAI.
    Returns:
    - int (0,1,2,3) if there is a valid prediction
    - None if the model blocks or does not return text
    """
    try:
        response = client.chat.completions.create(
            model=MODEL,
            messages=[
                {"role": "user", "content": build_prompt(text, task)}
            ],
            # GPT 5 mini only accepts temperature=1
            temperature=0 if MODEL != "gpt-5-mini" else 1,
            # GPT-5 Mini includes reasoning tokens in `max_completion_tokens`,
            # so increase the number of tokens to 500
            max_completion_tokens=5 if MODEL != "gpt-5-mini" else 500
        )

        output_text = response.choices[0].message.content

        if not output_text:
            print(f"No text was returned: {response}")
            return None

        match = re.search(r"[0-3]", output_text)
        return int(match.group()) if match else None

    except Exception as e:
        print(f"Error during classification: {e}")
        return None

In [ ]:
# Apply classification to the selected dataset
# The prediction is stored in a new column "pred"
print(f"Starting prediction for dataset: {task} | Total texts: {len(df)}")
df["pred"] = df["texto"].progress_apply(lambda x: classify_text(x, task))

Iniciando predicción para el dataset: anx | Total textos: 500


100%|██████████| 500/500 [05:20<00:00,  1.56it/s]


In [ ]:
# Second attempt for texts that were classified as NaN
mask = df["pred"].isna()

print(f"Pending Texts: {mask.sum()}")

df.loc[mask, "pred"] = (
    df.loc[mask, "texto"]
      .progress_apply(lambda x: classify_text(x, task))
)

### Predictions check


In [ ]:
from sklearn.metrics import f1_score, accuracy_score

# Replace None with an invalid class so it counts as an error
pred_labels = df["pred"].fillna(-1)

# If it is binary classification
if task in ["anx", "dep", "ed"]:
    # Column with the actual label
    true_labels = df["bs"]
    f1 = f1_score(true_labels, pred_labels, average="binary")
else:
    # Column with the actual label
    true_labels = df["label_mc"]
    # Multiclass classification
    f1 = f1_score(true_labels, pred_labels, average="macro")

acc = accuracy_score(true_labels, pred_labels)

print("F1-score:", round(f1, 4))
print("Acc:", round(acc, 4))

F1-score: 0.967
Acc: 0.94


In [ ]:
# Number of blocks
num_blocked = df["pred"].isna().sum()
block_rate = num_blocked / len(df)

print("Blocks:", num_blocked)
print("Block rate:", round(block_rate, 4))

Bloqueos: 0
Tasa de bloqueo: 0.0


### Store predictions


In [ ]:
output_path = (
    f"/content/drive/MyDrive/tfg/corpusMentalRiskES/resultados/"
    f"{MODEL}_{task}.csv"
)
df.to_csv(output_path, index=False)

## Multiple classification for Tara@5


To analyze the stability of the predictions, TARa@5 will be applied on a stratified subset of the dataset. Each example is evaluated five times with the same model, reusing the first prediction and generating four additional executions, allowing the degree of consistency of the outputs to be measured.


In [ ]:
print(f"Starting TARa@5 for dataset: {task} | Texts: {len(df_tara5)}")

output_path = (
    f"/content/drive/MyDrive/tfg/corpusMentalRiskES/resultados/"
    f"{MODEL}_{task}_tara5.csv"
)

# Apply the same classification function 4 more times
# Each prediction is stored in a new column "pred_run_{num_run}"
for run in range(1, 5):
    col_name = f"pred_run{run+1}"

    if col_name in df_tara5.columns:
        print(f"{col_name} already exists, skipping")
        continue

    print(f"\nExecuting pred_run{run+1}...")

    df_tara5[col_name] = df_tara5["texto"].progress_apply(lambda x: classify_text(x, task))

    # Incremental save
    df_tara5.to_csv(output_path, index=False)

    print(f"Run {run+1} successfully saved")

Iniciando TARa@5 para dataset: multi | Textos: 250

Ejecutando pred_run2...


100%|██████████| 250/250 [03:38<00:00,  1.14it/s]


Run 2 guardada correctamente

Ejecutando pred_run3...


100%|██████████| 250/250 [03:49<00:00,  1.09it/s]


Run 3 guardada correctamente

Ejecutando pred_run4...


100%|██████████| 250/250 [03:09<00:00,  1.32it/s]


Run 4 guardada correctamente

Ejecutando pred_run5...


100%|██████████| 250/250 [03:26<00:00,  1.21it/s]

Run 5 guardada correctamente
